<a href="https://colab.research.google.com/github/fuadsaniibrahim/FIP-Intern-Deliverable/blob/main/Module2_RL_DQN_CartPole.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2 (Part B): Reinforcement Learning with a Deep Q-Network (DQN)

**Technique:** Deep Q-Learning implemented from scratch in PyTorch (replay buffer, target network, epsilon-greedy exploration), applied to the CartPole-v1 balancing task.

**Task:** a pole is hinged on a cart. At each step the agent pushes the cart left or right. Reward is +1 per step the pole stays up, and an episode ends when the pole falls too far, the cart leaves the track, or 500 steps are reached. A score of 500 is a perfect episode.

**What this notebook does**
1. Measures a random-policy baseline
2. Implements and trains DQN (3 seeds)
3. Runs an ablation: the same DQN **without** the target network
4. Evaluates all agents on fresh episodes the agent never trained on
5. Renders the agent as an animation and wraps it in a small Gradio web app for the screen recording



In [ ]:
import os, io, copy, random, base64, time, warnings
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")      # headless rendering
warnings.filterwarnings("ignore")
from collections import deque
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import gymnasium as gym
import imageio
from PIL import Image

QUICK_RUN = False          # True = tiny dry run for debugging only. Leave False for real results.
EPISODES  = 500             # training episodes per run
N_SEEDS   = 3         # independent runs per variant
EVAL_EPISODES = 10 if QUICK_RUN else 50

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

print("gymnasium", gym.__version__, "| torch", torch.__version__, "| quick run:", QUICK_RUN)

## 1. Random-policy baseline
Any learned agent has to beat this, so it is measured first. Evaluation uses seeds 10,000 and above, which are never used during training.

In [ ]:
def evaluate(policy, n=EVAL_EPISODES, seed0=10_000):
    env = gym.make("CartPole-v1"); out = []
    for i in range(n):
        s, _ = env.reset(seed=seed0 + i); done = False; total = 0
        while not done:
            s, r, term, trunc, _ = env.step(policy(s)); done = term or trunc; total += r
        out.append(total)
    env.close(); return np.array(out)

_env = gym.make("CartPole-v1")
random_scores = evaluate(lambda s: _env.action_space.sample())
print(f"random policy: mean {random_scores.mean():.1f}, std {random_scores.std():.1f}, max {random_scores.max():.0f}")

## 2. DQN from scratch

**Ideas implemented**
* **Q-network:** maps a state (cart position, cart velocity, pole angle, pole angular velocity) to one value per action, the expected future reward of taking that action.
* **Experience replay:** transitions are stored and sampled randomly in mini-batches, which breaks the correlation between consecutive steps.
* **Target network:** a slowly updated copy of the Q-network provides the learning target, so the network is not chasing its own moving estimate.
* **Epsilon-greedy exploration:** random actions at the start, decaying towards mostly greedy actions.
* **Terminal handling:** only a true termination (pole fell) stops bootstrapping. Hitting the 500-step time limit is a truncation, not a failure, so the future value is still used.
* **Huber loss and gradient clipping** for stability.

### Worked example of one learning target (by hand)
Suppose a transition has reward r = 1, discount gamma = 0.99, the target network says the best next-state value is max Q'(s') = 45.2, and the Q-network currently says Q(s, a) = 44.0.

```
y        = r + gamma * max Q'(s')          (not terminal)
         = 1 + 0.99 * 45.2
         = 1 + 44.748
         = 45.748

delta    = y - Q(s, a)
         = 45.748 - 44.000
         = 1.748

Huber loss with threshold 1:
  |delta| = 1.748 > 1, so use the linear branch
  L       = |delta| - 0.5
          = 1.748 - 0.5
          = 1.248

If the transition ended the episode (pole fell), there is no future:
  y = r = 1
```
Each training step averages this loss over a batch of 64 transitions and takes one Adam step on the Q-network.

In [ ]:
class QNet(nn.Module):
    def __init__(self, n_obs=4, n_act=2, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_obs, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, n_act))
    def forward(self, x): return self.net(x)

class Replay:
    def __init__(self, capacity=50_000): self.buf = deque(maxlen=capacity)
    def push(self, *t): self.buf.append(t)
    def sample(self, n):
        s, a, r, s2, d = map(np.array, zip(*random.sample(self.buf, n)))
        f = lambda x, dt: torch.tensor(x, dtype=dt)
        return (f(s, torch.float32), f(a, torch.int64).unsqueeze(1), f(r, torch.float32),
                f(s2, torch.float32), f(d, torch.float32))
    def __len__(self): return len(self.buf)

def train_dqn(episodes=EPISODES, use_target=True, seed=0, lr=1e-3, gamma=0.99, batch=64,
              eps_start=1.0, eps_end=0.05, eps_decay=5000, warmup=1000, target_every=500, verbose=False):
    set_seed(seed)
    env = gym.make("CartPole-v1")
    q = QNet(); tgt = copy.deepcopy(q)
    opt = torch.optim.Adam(q.parameters(), lr=lr)
    buf, steps, rewards = Replay(), 0, []
    best_avg, best_state = -1, copy.deepcopy(q.state_dict())
    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep); total, done = 0, False
        while not done:
            eps = eps_end + (eps_start - eps_end) * np.exp(-steps / eps_decay)
            if random.random() < eps:
                a = env.action_space.sample()
            else:
                with torch.no_grad(): a = int(q(torch.tensor(s, dtype=torch.float32)).argmax())
            s2, r, term, trunc, _ = env.step(a); done = term or trunc
            buf.push(s, a, r, s2, float(term))           # only true termination stops bootstrapping
            s, total, steps = s2, total + r, steps + 1
            if len(buf) >= warmup:
                S, A, R, S2, D = buf.sample(batch)
                with torch.no_grad():
                    nxt = (tgt if use_target else q)(S2).max(1).values
                    y = R + gamma * nxt * (1 - D)
                loss = F.smooth_l1_loss(q(S).gather(1, A).squeeze(1), y)
                opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(q.parameters(), 10); opt.step()
                if use_target and steps % target_every == 0: tgt.load_state_dict(q.state_dict())
        rewards.append(total)
        avg = np.mean(rewards[-20:])
        if len(rewards) >= 20 and avg > best_avg: best_avg, best_state = avg, copy.deepcopy(q.state_dict())
        if verbose and (ep + 1) % 50 == 0: print(f"  ep {ep+1:3d}  reward {total:5.0f}  avg20 {avg:6.1f}  eps {eps:.2f}")
    env.close(); q.load_state_dict(best_state); q.eval()
    return q, rewards, best_avg

def greedy(q): return lambda s: int(q(torch.tensor(s, dtype=torch.float32)).argmax())

## 3. Train DQN with a target network (several seeds)

In [ ]:
runs = {}
t0 = time.time()
for seed in range(N_SEEDS):
    q, rw, best = train_dqn(use_target=True, seed=seed, verbose=True)
    runs[("target", seed)] = dict(model=q, rewards=rw, best_train_avg=best, eval=evaluate(greedy(q)))
    print(f"seed {seed}: eval mean {runs[('target', seed)]['eval'].mean():.1f}  ({time.time()-t0:.0f}s elapsed)")

## 4. Ablation: the same DQN without a target network

In [ ]:
for seed in range(N_SEEDS):
    q, rw, best = train_dqn(use_target=False, seed=seed)
    runs[("no_target", seed)] = dict(model=q, rewards=rw, best_train_avg=best, eval=evaluate(greedy(q)))
    print(f"seed {seed}: eval mean {runs[('no_target', seed)]['eval'].mean():.1f}  ({time.time()-t0:.0f}s elapsed)")

## 5. Learning curves

In [ ]:
def smooth(x, k=20): return np.convolve(x, np.ones(k)/k, mode="valid") if len(x) >= k else np.array(x)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for a, (key, title) in zip(ax, [("target", "DQN with target network"), ("no_target", "DQN without target network")]):
    for seed in range(N_SEEDS):
        r = runs[(key, seed)]["rewards"]
        a.plot(r, alpha=0.2); a.plot(range(19, 19 + len(smooth(r))), smooth(r), label=f"seed {seed}")
    a.axhline(random_scores.mean(), color="gray", ls="--", label="random policy")
    a.set_title(title); a.set_xlabel("training episode"); a.legend()
ax[0].set_ylabel("episode reward (faint) and 20-episode average")
plt.tight_layout(); plt.savefig("learning_curves.png", dpi=130); plt.show()

## 6. Evaluation on unseen episodes
Greedy policy (no exploration), 50 fresh episodes per run, on seeds never used for training. The summary table is built from the actual runs above.

In [ ]:
rows = []
for (key, seed), r in runs.items():
    rows.append(dict(agent="DQN + target network" if key == "target" else "DQN, no target network",
                     seed=seed, eval_mean=r["eval"].mean(), eval_min=r["eval"].min(), eval_max=r["eval"].max()))
per_run = pd.DataFrame(rows)
summary = (per_run.groupby("agent")["eval_mean"].agg(["mean", "std", "min", "max"]).round(1))
summary.loc["Random policy"] = [random_scores.mean(), random_scores.std(), random_scores.min(), random_scores.max()]
summary = summary.round(1).rename(columns={"mean": "mean reward", "std": "std across seeds", "min": "worst run", "max": "best run"})
display(per_run.round(1)); display(summary)
per_run.to_csv("rl_per_run_results.csv", index=False); summary.to_csv("rl_summary.csv")

## 7. Watch the agent

In [ ]:
def run_episode_frames(policy, seed=0, max_frames=250, stride=2):
    env = gym.make("CartPole-v1", render_mode="rgb_array")
    s, _ = env.reset(seed=seed); done = False; total = 0; frames = []; t = 0
    while not done:
        s, r, term, trunc, _ = env.step(policy(s)); done = term or trunc; total += r
        if t % stride == 0 and len(frames) < max_frames:
            frames.append(np.array(Image.fromarray(env.render()).resize((300, 200))))
        t += 1
    env.close(); return frames, total

# pick the demo model: the run with the best training-time average (not the best test score)
best_key = max([k for k in runs if k[0] == "target"], key=lambda k: runs[k]["best_train_avg"])
demo_model = runs[best_key]["model"]
frames, score = run_episode_frames(greedy(demo_model), seed=123)
imageio.mimsave("dqn_agent.gif", frames, duration=40, loop=0)
print(f"demo agent (seed {best_key[1]}): episode reward {score:.0f}, {len(frames)} frames saved to dqn_agent.gif")
from IPython.display import Image as IPImage, display
display(IPImage("dqn_agent.gif"))

In [ ]:
torch.save(demo_model.state_dict(), "dqn_cartpole.pt")
print("saved dqn_cartpole.pt")

## 8. Working application (Gradio)
A small web app: pick the trained agent or the random policy, choose an episode seed, and watch the result. Run the cell below and use the public link for the screen recording.

In [ ]:
import gradio as gr

def gif_html(frames):
    buf = io.BytesIO(); imageio.mimsave(buf, frames, format="GIF", duration=40, loop=0)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f'<img src="data:image/gif;base64,{b64}" style="max-width:100%;border-radius:8px"/>'

def play(agent, seed):
    seed = int(seed)
    if agent == "Trained DQN":
        pol = greedy(demo_model)
    else:
        rng = np.random.RandomState(seed); pol = lambda s: int(rng.randint(2))
    frames, score = run_episode_frames(pol, seed=seed)
    verdict = "perfect episode" if score >= 500 else f"pole fell after {int(score)} steps"
    return gif_html(frames), f"{agent}: reward {int(score)} ({verdict}). Maximum possible is 500."

demo = gr.Interface(
    fn=play,
    inputs=[gr.Radio(["Trained DQN", "Random policy"], value="Trained DQN", label="Agent"),
            gr.Slider(0, 999, value=7, step=1, label="Episode seed")],
    outputs=[gr.HTML(label="Episode"), gr.Textbox(label="Result")],
    title="CartPole: Deep Q-Network agent",
    description="Compare a DQN trained from scratch with a random policy on the same starting conditions.",
    flagging_mode="never")

if QUICK_RUN:
    print(play("Trained DQN", 7)[1])          # dry run: call the function directly
else:
    demo.launch(share=True)

## 9. Findings

In [ ]:
t = summary
dqn, nt, rnd = t.loc["DQN + target network"], t.loc["DQN, no target network"], t.loc["Random policy"]
print(f"Random policy averaged {rnd['mean reward']} reward per episode.")
print(f"DQN with a target network averaged {dqn['mean reward']} (std {dqn['std across seeds']} across {N_SEEDS} seeds, worst run {dqn['worst run']}, best run {dqn['best run']}).")
print(f"DQN without a target network averaged {nt['mean reward']} (std {nt['std across seeds']}, worst run {nt['worst run']}, best run {nt['best run']}).")
print(f"Improvement of DQN over random: {dqn['mean reward'] / rnd['mean reward']:.1f}x.")

### Discussion
* **Did it learn?** Compare the DQN rows with the random policy in the table above. The gap is the learning, since both are scored on the same unseen episodes.
* **Variance across seeds is the main caveat.** Reinforcement learning results depend heavily on the random seed, which is why three runs are shown and the spread is reported. A single run, however good, would not be trustworthy evidence.
* **Target network ablation.** The two learning-curve panels and the table show how the variant without a target network behaved over the same seeds. With only three seeds per variant this is illustrative, not a statistically conclusive test. The standard argument is that bootstrapping from a moving copy of the same network can destabilise learning, which a frozen target is meant to prevent.
* **Model choice for the demo** used the training-time moving average, not the evaluation score, so the demo agent was not picked by looking at the test episodes.

### Limitations
* CartPole is a small, low-dimensional task. The same method does not transfer unchanged to harder environments, which usually need improvements such as double DQN, prioritised replay, or policy-gradient methods like PPO.
* Hyperparameters were set by a short manual search, not an exhaustive one.
* Three seeds per variant is a small sample.